# ഹ്യൂമൻ-ഇൻ-ദി-ലൂപ്: പ്രീ-ആക്ഷൻ ഗേറ്റുകൾ, റിസ്ക് ടിയറിംഗ്, ആഡിറ്റ് ലോങ്ങിംഗ്

ഈ പാഠത്തിനുള്ള README ഹ്യൂമൻ-ഇൻ-ദി-ലൂപിനെ പരിചയപ്പെടുത്തുന്നു ഒരു ചെറു സ്നിപ്പെറ്റിലൂടെ, ഏജന്റ് ഇതിനകം പ്രതികരണം നൽകിയതിന് ശേഷം ഉപഭോക്താവിന് `APPROVE` അല്ലെങ്കിൽ `REJECT` എന്ന് ചോദിക്കുന്നത്. ആ പാറ്റേൺ നല്ല തുടക്കമാണ്, പക്ഷേ പ്രൊഡക്ഷൻ HITL നടപ്പാക്കലുകൾ പൊതുവെ മൂന്ന് അധിക ഘടകങ്ങൾ ആവശ്യപ്പെടുന്നു:

1. ഏജന്റ് ഒരു റിസ്ക് വഹിക്കുന്ന ഘട്ടം നടത്തുന്നതിന് **മുൻപ്** പ്രവർത്തിക്കുന്ന ഒരു **പ്രീ-ആക്ഷൻ ഗേറ്റ്**, അങ്ങനെ ചെലവ്, പ്രവൃത്തി പരിമിതിയും, വൈകിപ്പിടിപ്പും നിയന്ത്രണത്തിൽ നിർത്താൻ സാധിക്കും.
2. **റിസ്ക് ടിയറിംഗ്**, അങ്ങനെ കുറഞ്ഞ റിസ്ക് ഉള്ള പ്രവർത്തനങ്ങൾ സ്വയം നടപ്പാക്കുക, മധ്യരേഖാ റിസ്ക് ഉള്ള പ്രവർത്തനങ്ങൾ ബാച്ചായി അംഗീകരിക്കുക, ഉയർന്ന റിസ്ക് ഉള്ള പ്രവർത്തനങ്ങൾ മാത്രമേ മനുഷ്യനെ തടയുകയുള്ളൂ.
3. ഓരോ ഗേറ്റ് തീരുമാനവും JSONL ആയി രേഖപ്പെടുത്തുന്ന ഒരു **ആഡിറ്റ് ലോഗും പരിഷ്കാര ചക്കവും**, ഇനി `Revising...` മാത്രം പ്രിന്റ് ചെയ്യുന്നതിന് പകരം അപ്ഡേറ്റിനു കാരണമുള്ള ഘടകം നൽകിക്കൊണ്ട് ഏജന്റിനെ വീണ്ടും പ്രോംപ്റ്റ് ചെയ്യുക.

ഈ നോട്ട്‌బുക്ക് `06-system-message-framework.ipynb` ൽ ഉപയോഗിച്ചിട്ടുള്ള സമാന പ്രിമിറ്റിവുകൾ ഉപയോഗിച്ചാണ് ഓരോ ഘടകവും നിർമ്മിക്കുന്നത്. ഇത് `DEMO_MODE = True` ആയപ്പോൾ (ഇന്ററാക്ടീവ് ഇൻപുട്ട് വേണ്ട)യും അല്ലെങ്കിൽ `DEMO_MODE = False` ആയപ്പോൾ യഥാർത്ഥ `input()` പ്രോംപ്റ്റുകൾ ഉപയോഗിച്ച് അവസാനത്തേയ്ക്ക് പ്രവർത്തിക്കുന്നു. ശ്രദ്ധിക്കുക: DEMO_MODE ൽ മൂന്നാം ലക്ഷ്യത്തിന് വീണ്ടും ശ്രമം സ്ക്രിപ്റ്റ് ചെയ്തതാണ്, അതിനാൽ ലൂപ് പ്രവർത്തനം മുഴുവൻ കാണപ്പെടുന്നു. യഥാർത്ഥ പരിഷ്കാര നിയന്ത്രിത പുന:ശ്രേണീകരണം `DEMO_MODE = False` ആയും ഒപ്പറേറ്ററും ആവശ്യമാണ്.

**പരിധി പുറത്ത് (മറ്റു പാഠങ്ങളിൽ കൈകാര്യം ചെയ്തിരിക്കുന്നു):** പ്രവർത്തക സംരക്ഷണം, ആക്സസ് നിയന്ത്രണം (പാഠം 06 README ഭീഷണി 2), ടൂൾ-കോൾ മിഡിൽവെയർ (പാഠം 14 MAF ഡീപ്പ് ഡൈവ്), മൾട്ടി-ഏജന്റ് വിവാദ പാറ്റേണുകൾ.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## പാറ്റേൺ 1: മുൻ-കയ്യടി ഗേറ്റ്

README-യിലെ HITL സ്നിപ്പറ്റ് ഏജന്റിനെ ആദ്യമായി വിളിച്ച്, പിന്നീട് ഫലത്തെ അംഗീകരിക്കാൻ ഉപയോക്താവിനെ ചോദിക്കുന്നു. അത് ഒരു **പോസ്റ്റ്-ആക്ഷൻ** പ്രവാഹമാണ്. ഏജന്റ് ഇതിനകം നടപ്പാക്കിയിട്ടുണ്ട്, അതിനാൽ LLM കോളിന്റെ ചെലവ് ഇതിനകം നൽകിയതായിരിക്കും, കൂടാതെ ഏതെങ്കിലും വിഭാജ്യഫലം (ഇമെയിൽ അയച്ചത്, ഡേറ്റാബേസ് റോ എഴുതിയത്, കമന്റ് പോസ്റ്റ് ചെയ്തത്) ഇതിനകം സംഭവിച്ചിട്ടുണ്ട്.

**മുൻ-ആക്ഷൻ** പ്രവാഹം ഏജന്റ് അപകടം തീർക്കുന്ന ഘട്ടം മുന്നറിയിപ്പായി ഗേറ്റ് ഇടുന്നു. ഏജന്റ് പ്രവർത്തനം നിർദ്ദേശിക്കുന്നു, ഗേറ്റ് പ്രവൃത്തി നടത്തുമോ എന്നത് തീരുമാനിക്കുന്നു, അംഗീകാരം കിട്ടിയാൽ മാത്രമേ വിഭാജ്യഫലം സംഭവിക്കുകയുള്ളു.

| സവിശേഷത | പോസ്റ്റ്-ആക്ഷൻ അംഗീകാരം (README സ്നിപ്പറ്റ്) | മുൻ-ആക്ഷൻ ഗേറ്റ് (ഈ നോട്ട്‌ബുക്ക്) |
|---|---|---|
| അംഗീകാരം എപ്പോൾ നടക്കുന്നു? | ഏജന്റ് ഫലം നൽകിയശേഷം | ഏത് വക青青草വും നടപ്പിലാക്കുന്നതിനു മുമ്പ് |
| നിരസിക്കൽ സമയത്തെ LLM ചെലവ് | ഇതിനകം നൽകിയിട്ടുണ്ട് | നിർദ്ദേശത്തിനായുള്ള മാത്രമാണ് ചെലവ്, പ്രവർത്തനത്തിനല്ല |
| മടങ്ങിക്കൊണ്ടുപോകാനാവാത്ത വിഭാജ്യഫലങ്ങൾ | സാധ്യതയുണ്ട് (പ്രവൃത്തി ഇതിനകം നടന്ന് കഴിഞ്ഞു) | തടയുന്നു |
| ഓഡിറ്റ് വ്യക്തമാക്കൽ | അംഗീകാരം പ്രിന്റ് സ്റ്റേറ്റ്മെന്റാണ് | അംഗീകാരം ടൈംസ്റ്റാമ്പ്, പ്രവർത്തി, കാരണം എന്നിവയുള്ള JSON റെക്കോർഡാണ് |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## മാതൃകം 2: അപകട സ്തരീകരണം

എല്ലാ പ്രവർത്തനത്തിനും മനുഷ്യ അംഗീകാരം ആവശ്യമില്ല. ഒരു പബ്ലിക് API-വിൽ വേണ്ടാത്ത വായന മാത്രമായുള്ള നോക്കപ്പ് ചർച്ചയുടെ തലത്തിൽ വ്യത്യസ്തമാണ്, ഉപഭോക്തൃ ഇമെയിൽ അയക്കുന്നതുമായി താരതമ്യം ചെയ്യുമ്പോൾ. രണ്ടും ഒരുപോലെ പരിഗണിക്കുന്നത് ഓപ്പറേറ്ററുടെ ശ്രദ്ധ മുടക്കുകയും ഏജന്റ് മന്ദഗതിയാക്കുകയും ചെയ്യുന്നു.

ഒരു ലളിതമായ 3-സ്തർ മാതൃകം:

| പട്ടിക | ഉദാഹരണങ്ങൾ | അംഗീകാരം പ്രവാഹം |
|---|---|---|
| `താഴ്ന്ന` (വായന മാത്രം) | ഒരു അറിവ് അടിസ്ഥാനത്തിലുള്ള തിരയൽ, വിമാന ഓപ്ഷനുകൾ നോക്കുക, ഒരു പൊതു വെബ് പേജ് സ്വാധീനം കൊള്ളുക | ഓട്ടോമാറ്റിക് നിർവഹണം, ഓഡിറ്റ് വേണ്ടി ലോഗ് ചെയ്യപ്പെട്ടത് |
| `മധ്യസ്ഥ` (സാധാരണ മാറ്റം) | ഫലം കാഷ് ചെയ്യുക, കൗണ്ടർ വർധിപ്പിക്കുക, ഒരു ഓർമ്മപ്പെടുത്തൽ നിശ്ചയിക്കുക | ഓട്ടോമാറ്റിക് നിർവഹണം, പക്ഷേ പ്രതിദിന അവലോകനവുമായി ബാച്ച് ചെയ്യുന്നു |
| `ഉയർന്ന` (ബാഹ്യപ്രവേശനമായോ അല്ലെങ്കിൽ തിരിയാതെ പോകുന്നോ) | ഇമെയിൽ അയക്കുക, കാർഡ് ചാർജ് ചെയ്യുക, ഒരു പബ്ലിക് ചാനലിൽ പോസ്റ്റ് ചെയ്യുക | മനുഷ്യ അംഗീകാരത്തിലേക്ക് തടസമുണ്ടാക്കുക |

ഇതാണ് ഒരു സ്തരീകരണം. ഉൽപാദന სისტემകൾ പലപ്പോഴും കൂടുതൽ സൂക്ഷ്മമായ സ്തറിക്ഷേത്രങ്ങൾ ഉപയോഗിക്കുന്നു (ഉദാ: AWS IAM അനുമതി നിലകൾ, റോൾ അധിഷ്ഠിത ആക്സസ് സ്തറുകൾ). താഴെ കൊടുത്തിരിക്കുന്ന 3-സ്തർ പതിപ്പ് വായന മാത്രം അന്തരീക്ഷവും സൈഡ്-ഇഫക്റ്റ് പ്രവർത്തനവും (side-effecting actions) എടുക്കുന്ന ഏജന്റിനായുള്ള ഏറ്റവും ചെറിയ പ്രയോജനപ്രദമായ പതിപ്പാണ്.

താഴെ കൊടുത്ത ക്ലാസിഫയർ കീവേഡ് ഹ്യൂറിസ്റ്റിക്‌സ് ഉപയോഗിക്കുന്നു, അതിനാൽ ഡെമോ നിശ്ചിതവും ചെലവുകൊള്ളവുമാണ്. ഉൽപാദന സംവിധാനത്തിൽ, നിങ്ങൾ ഇത് പഠിച്ച ക്ലാസിഫയറോ നയം എഞ്ചിനോ ഉപയോഗിച്ച് മാറ്റിക്കൊൾക്കും.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## പാറ്റേൺ 3: ഓഡിറ്റ് ലോഗ്‌വും തിരുത്തൽ ലൂപ്പും

ഒരു `print("Response approved.")` ഓഡിറ്റ് ലോഗ് അല്ല. വിശ്വാസത്തിനായി, ഓരോ ഗോൾ തീരുമാനവും പിന്നീട് നിങ്ങൾ ചോദ്യം ചെയ്യാനും, വീണ്ടും പ്ലേ ചെയ്യാനും, അല്ലെങ്കിൽ ഒരു സംഭവ പഠനത്തിന് അറ്റാച്ച് ചെയ്യാനുമുള്ള ഘടിതമായ സംവേദനമായി രേഖപ്പെടുത്തണം.

രണ്ടു ഭാഗങ്ങൾ:

1. **അപ്പ്‌എൻലി JSONL.** ഓരോ തീരുമാനത്തിനും ഒരു വരി, ടൈംസ്റ്റാമ്പ്, പ്രവർത്തനം, തിയർ, തീരുമാനം, കാരണം എന്നിവയുള്ളത്. grep ചെയ്യാൻ എളുപ്പം, പിന്നീട് യഥാർത്ഥ ലോഗ് സ്റ്റോറിലേക്ക് അയയ്ക്കാനും എളുപ്പം.
2. **തിരസ്കരണത്തിൽ തിരുത്തൽ ലൂപ്.** ഗേറ്റ് `deny` തിരിച്ചുവിട്ടാൽ, ഏജന്റ് താൻ തന്നെ തിരസ്കരണ കാര്യം കോണ്ടക്റ്റിൽ വെച്ച് വീണ്ടും പ്രോംപ്റ്റ് ചെയ്യുന്നു, അങ്ങനെ അടുത്ത നിർദ്ദേശം പ്രശ്നം ഒഴിവാക്കാനാകും.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## അധിക വിഭവങ്ങൾ

ചില മറ്റ് പൊതു പ്രോജക്റ്റുകൾ ഈ HITL പാറ്റേണുകളുടെ വ്യത്യസ്ത രൂപങ്ങൾ നടപ്പിലാക്കുന്നു. നിങ്ങളുടെ സ്റ്റാക്കിന് അനുയോജ്യമായത് കണ്ടെത്താൻ സമീപനങ്ങൾ താരതമ്യം ചെയ്‌യുക:

- **LangChain** ഹ്യൂമൻ-ഇൻ-ദ-ലൂപ്പ് ടൂൾ റാപ്പറുകൾ ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): മനുഷ്യന്റെ ഇൻപുട്ട് കാത്തുനിൽക്കുന്നതിലൂടെ പ്രവർത്തനം തടയുന്ന ഡ്രോപ്പിൻ ടൂൾ റാപ്പറുകൾ.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ഇത് പുനഃസംഘടിപ്പിച്ചിരിക്കുന്നു): ബഹുജന-ഏജന്റ് സംവാദങ്ങളിൽ മനുഷ്യനെ പ്രതിനിധാനം ചെയ്യുന്നതിന് പ്രത്യേകമായി ഏജന്റ് സ്ഥാനം ഉപയോഗിക്കുന്നു.
- **Microsoft Agent Framework (MAF)** ഫംഗ്ഷൻ-ഇൻവൊക്കേഷൻ മിഡിൽവെയർ ([docs](https://learn.microsoft.com/agent-framework/)): എല്ലാ ടൂൾ/ഫംഗ്ഷൻ کالിനും ചുറ്റുമുള്ള മിഡിൽവെയർ, ഗേറ്റിംഗ് ലജിക്കും അംഗീകരണപ്രവാഹങ്ങൾക്കും അനുയോജ്യം.

ഓരോ പ്രോജക്റ്റും മൂന്ന് ഉപ-പാറ്റേണുകളും വ്യത്യസ്തമായി കൈവരിക്കുന്നു: LangChain അവയെ ടൂളുകളായി റാപ്പ് ചെയ്യുന്നു, AutoGen ഏജന്റ് സ്ഥാനം ഉപയോഗിക്കുന്നു, Microsoft Agent Framework ഫംഗ്ഷൻ-ഇൻവൊക്കേഷൻ മിഡിൽവെയർ ഉപയോഗിക്കുന്നു. നിങ്ങളുടെ സ്വന്തം ഏജന്റിനായി ഒരു ഡിസൈൻ തിരഞ്ഞെടുക്കുന്നതിന് മുമ്പ് ഒരു അല്ലെങ്കിൽ രണ്ട് നടപ്പിലാക്കൽ സമ്പൂർണ്ണമായി വായിക്കുക.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
